In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph

# Vector Stores
!pip install chromadb
!pip install langchain-chroma

# Import the Python SDK for Colab secrets
from google.colab import userdata


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 25.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [2]:
import json
import os
import time
from operator import itemgetter  # Used in exercises/solutions for LCEL key extraction
# LangChain is a framework for building LLM applications
# Reference: https://python.langchain.com/docs/get_started/introduction
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI integrations
from langchain_chroma import Chroma  # Vector database for similarity search
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smart text chunking
from langchain_core.documents import Document  # Document abstraction
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.messages import HumanMessage, AIMessage  # Chat history message types
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain_core.runnables import RunnablePassthrough  # Pass data through pipeline> lambda x:x

In [3]:
# ============================================================================
# PART 1: Data Ingestion & Vector Store Setup
# ============================================================================
print("\n" + "="*80)
print("PART 1: Data Ingestion Pipeline")
print("="*80)

# Load tickets
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Convert to LangChain Document objects
# Documents are the core abstraction in LangChain - they combine content with metadata
# Reference: https://python.langchain.com/docs/modules/data_connection/document_loaders/
documents = []
for ticket in tickets:
    # Create rich document with all context
    # TIP: Structure your content logically - LLMs understand formatted text better
    content = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Date: {ticket['created_date']} to {ticket['resolved_date']}

Problem Description:
{ticket['description']}

Resolution:
{ticket['resolution']}
    """.strip()

    # Create Document with metadata
    # Metadata is crucial for filtering, citation, and source tracking
    # Best practice: Include all information you might want to filter or display later
    doc = Document(
        page_content=content,  # The actual text content
        metadata={  # Structured data about the document
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'source': f"Ticket {ticket['ticket_id']}"
        }
    )
    documents.append(doc)

print(f"✓ Created {len(documents)} documents with metadata")


PART 1: Data Ingestion Pipeline
✓ Loaded 22 support tickets
✓ Created 22 documents with metadata


In [4]:
# Initialize OpenAI embeddings
# Embeddings convert text into vectors for semantic search
# Reference: https://platform.openai.com/docs/guides/embeddings

# Set your OpenAI API key from Colab secrets
# IMPORTANT: Add your OpenAI API key to Colab secrets under the name 'OPENAI_API_KEY'.
# Click on the 'key' icon in the left sidebar -> Secrets -> Add new secret
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_APIKEY')

print("\nInitializing OpenAI embedding model...")
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
)
print("✓ OpenAI embedding model ready")

# Build vector store using Chroma
# Chroma is an open-source vector database optimized for AI applications
# It stores embeddings and enables fast similarity search
# Reference: https://docs.trychroma.com/
print("\nBuilding Chroma vector store...")
# Always rebuild from scratch so repeated runs don't accumulate duplicate documents.
import shutil
persist_directory = "./rag_vectorstore"
if os.path.exists(persist_directory):
    try:
        shutil.rmtree(persist_directory)
    except PermissionError:
        # Windows may keep Chroma files locked briefly from a prior run.
        # Fall back to a unique directory so the demo can continue.
        persist_directory = f"./rag_vectorstore_{int(time.time())}"
        print(f"⚠ Vector store directory is locked; using {persist_directory} instead")
vector_store = Chroma.from_documents(
    documents=documents,  # Our support ticket documents
    embedding=embeddings,  # Embedding function to use
    collection_name="supportdesk_rag",  # Name for this collection
    persist_directory=persist_directory,  # Where to save the database
    collection_metadata={"hnsw:space": "cosine"}  # Use cosine distance for similarity search
)
print("✓ Vector store created and persisted")



Initializing OpenAI embedding model...
✓ OpenAI embedding model ready

Building Chroma vector store...
✓ Vector store created and persisted


In [5]:
# ============================================================================
# PART 2: Create Retriever
# ============================================================================
print("\n" + "="*80)
print("PART 2: Setting Up Retriever")
print("="*80)

# Create a retriever from the vector store
# Retrievers are the interface for querying the vector store
# Reference: https://python.langchain.com/docs/modules/data_connection/retrievers/
retriever = vector_store.as_retriever(
    search_type="similarity",  # Use cosine similarity for ranking
    search_kwargs={"k": 3}  # Retrieve top-3 most similar documents
    )
    # Other options:
    # - "mmr" (Maximal Marginal Relevance): Balances relevance with diversity
        # retriever = vector_store.as_retriever(
        # search_type="mmr",
        # search_kwargs={
        #     "k": 3,                # final number of docs returned
        #     "fetch_k": 10,         # candidates fetched before MMR re-ranks
        #     "lambda_mult": 0.5     # 0 = max diversity, 1 = max relevance
        # }
        # )
    # - "similarity_score_threshold": Only return docs above a score threshold
        # retriever = vector_store.as_retriever(
        # search_type="similarity_score_threshold",
        # search_kwargs={
        #     "score_threshold": 0.7,  # minimum similarity score (0-1)
        #     "k": 3                   # max docs to return
        # }
        # )


print("✓ Retriever configured:")
print(f"  - Search type: similarity")
print(f"  - Top-K results: 3")
print("\nTIP: k=3-5 is usually optimal. Too few → missing context, too many → noise")



PART 2: Setting Up Retriever
✓ Retriever configured:
  - Search type: similarity
  - Top-K results: 3

TIP: k=3-5 is usually optimal. Too few → missing context, too many → noise


In [6]:
# ============================================================================
# PART 3: Initialize LLM
# ============================================================================
print("\n" + "="*80)
print("PART 3: Initializing Language Model")
print("="*80)

# Check if OpenAI key is available
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key found")
    # Initialize ChatOpenAI for generation
    # Reference: https://python.langchain.com/docs/integrations/chat/openai
    llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )
    print(f"✓ Using {os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')}")
else:
    print("⚠ OpenAI API key not found!")
    print("  Please set OPENAI_API_KEY environment variable")
    print("  Or use Ollama: ollama pull llama2")
    print("\nFor this demo, we'll show the prompt without generating answers.")
    llm = None


PART 3: Initializing Language Model
✓ OpenAI API key found
✓ Using gpt-4o-mini


In [7]:
# ============================================================================
# PART 4: Build format_docs Helper Function
# ============================================================================
print("\n" + "="*80)
print("PART 4: Build format_docs Helper Function")
print("="*80)

# Helper function to format retrieved documents
# This concatenates all retrieved document contents into a single context string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single context string.

    Why this helper exists:
    - Retrievers return `List[Document]` objects.
    - Prompt templates expect plain strings for `{context}`.
    - Joining with separators keeps boundaries visible to the LLM.
    """
    # Keep document boundaries explicit so the model can attribute facts by chunk.
    return "\n\n---\n\n".join([doc.page_content for doc in docs])




PART 4: Build format_docs Helper Function


In [13]:

# ============================================================================
# PART 4: Build condense & conversastional prompt template
# ============================================================================
print("\n" + "="*80)
print("PART 4: Build prompt template")
print("="*80)

if llm:
    # ── Step 1: Query Reformulation ────────────────────────────────────
    # The retriever only sees the raw question string.
    # "What was the resolution for that ticket?" → retriever gets vague query.
    # Fix: Use the LLM to rewrite follow-ups into standalone queries first.
    condense_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "Given the chat history and a follow-up question, rephrase the "
         "follow-up as a standalone question that includes all necessary "
         "context from the history. If the question is already standalone, "
         "return it unchanged.\n\n"
         "IMPORTANT: Output ONLY the rephrased question. Do NOT answer it. "
         "Do NOT include any explanation or additional text."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])

    # ── Step 2: Conversation-aware prompt ──────────────────────────────
    # MessagesPlaceholder expands the history list into the prompt at call time
    conv_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are SupportDesk AI. Answer using the ticket context below and the chat history.
    Use chat history to resolve references like "that issue" or "that ticket".
    For factual claims, prioritize the retrieved context.
    If information is not available in context or history, say "I don't have that information."
    Always cite ticket IDs when available.

Context:
{context}"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ])


else:
    print("(LLM not configured — would run multi-turn conversation here)")
    print("\nKey pattern:\n")
    print("  history = []")
    print()
    print("  # Step 1: Rewrite follow-up into standalone query (skip if no history)")
    print("  if history:")
    print("      standalone = condense_chain.invoke({'question': q, 'chat_history': history})")
    print("  else:")
    print("      standalone = q")
    print()
    print("  # Step 2: Retrieve using standalone query, generate with original question")
    print("  context = format_docs(retriever.invoke(standalone))")
    print("  answer = (conv_prompt | llm | StrOutputParser()).invoke({")
    print("      'context': context, 'chat_history': history, 'question': q")
    print("  })")
    print()
    print("  history.append(HumanMessage(content=q))")
    print("  history.append(AIMessage(content=answer))")
    print()
    print("Query reformulation ensures the retriever gets meaningful queries")
    print("instead of vague pronouns like 'it' or 'that ticket'.")




PART 4: Build prompt template


In [14]:
# ============================================================================
# PART 5: Conversation with History (Multi-Turn RAG) function
# ============================================================================
print("\n" + "="*80)
print("PART 8: Conversation with History (Multi-Turn RAG)")
print("="*80)
print("""
Problem with single-turn RAG:
  Turn 1: "How do I fix authentication failures?"  → good answer
  Turn 2: "How long did it take to resolve?"       → loses context! "it" = ???

Solution: Two-part fix:
  1. MessagesPlaceholder injects prior HumanMessage / AIMessage objects into the prompt
     so the LLM can understand references like "that issue" or "it".
  2. Query reformulation rewrites follow-up questions into standalone queries
     BEFORE retrieval, so the retriever searches for the right documents.
     e.g. "How do I fix it?" + history → "How do I fix authentication failures?"
""")

def ask_with_history(question, history):
        """Ask a question with query reformulation and history tracking.

        On the first turn (empty history), skip the condense step and send
        the question directly to the retriever. On follow-up turns, rewrite
        the question into a standalone query so the retriever finds the
        right documents (e.g. "that ticket" → "TICK-001").
        """
        if not history:
            # First turn: no history to condense, retrieve directly
            standalone = question
        else:
            # Follow-up turn: rewrite using history context
            # e.g. "What was the resolution for that ticket?" →
            #      "What was the resolution for ticket TICK-001?"

            # Chain that rewrites the question: dict → standalone question string
            condense_chain = condense_prompt | llm | StrOutputParser()

            standalone = condense_chain.invoke({
                "question": question, "chat_history": history
            })

        # Show what the retriever actually sees (useful for debugging)
        if standalone != question:
            print(f"         [Rewritten query → {standalone}]")

        # Retrieve docs using the standalone query and generate the answer
        context = format_docs(retriever.invoke(standalone))

        # Chain that expands the history list into the prompt at call time
        conv_chain = conv_prompt | llm | StrOutputParser()

        answer =  conv_chain.invoke({
            "context": context,
            "chat_history": history,
            "question": question,  # Original question, not the rewritten one
        })

        history.append(HumanMessage(content=question))
        history.append(AIMessage(content=answer))
        return answer




PART 8: Conversation with History (Multi-Turn RAG)

Problem with single-turn RAG:
  Turn 1: "How do I fix authentication failures?"  → good answer
  Turn 2: "How long did it take to resolve?"       → loses context! "it" = ???

Solution: Two-part fix:
  1. MessagesPlaceholder injects prior HumanMessage / AIMessage objects into the prompt
     so the LLM can understand references like "that issue" or "it".
  2. Query reformulation rewrites follow-up questions into standalone queries
     BEFORE retrieval, so the retriever searches for the right documents.
     e.g. "How do I fix it?" + history → "How do I fix authentication failures?"



In [15]:


# ============================================================================
# PART 6: Test Conversation with History (Multi-Turn RAG) function
# ============================================================================

history = []

q1 = "How do I fix authentication failures after a password reset?"
print(f"Turn 1 — User: {q1}")
a1 = ask_with_history(q1, history)
print(f"         Assistant: {a1[:300]}{'...' if len(a1) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")

q2 = "What was the ticket ID for that issue?"
print(f"\nTurn 2 — User: {q2}")
a2 = ask_with_history(q2, history)
print(f"         Assistant: {a2[:300]}{'...' if len(a2) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")

q3 = "What was the resolution for that ticket?"
print(f"\nTurn 3 — User: {q3}")
a3 = ask_with_history(q3, history)
print(f"         Assistant: {a3[:300]}{'...' if len(a3) > 300 else ''}")
print(f"         [chat_history now has {len(history)} messages]")

print(f"\n✓ History contains {len(history)} messages ({len(history)//2} complete turns)")
print("TIP: In production, cap history to avoid token bloat:")
print("       history = history[-6:]  # keep last 3 turns")

Turn 1 — User: How do I fix authentication failures after a password reset?
         Assistant: To fix authentication failures after a password reset, you can follow the resolution steps from Ticket ID TICK-001. The issue was caused by an updated password hash algorithm while session tokens were not invalidated. Here’s what you can do:

1. Clear all active sessions for the affected users.
2. F...
         [chat_history now has 2 messages]

Turn 2 — User: What was the ticket ID for that issue?
         [Rewritten query → What was the ticket ID for the issue related to authentication failures after a password reset?]
         Assistant: The ticket ID for that issue is TICK-001.
         [chat_history now has 4 messages]

Turn 3 — User: What was the resolution for that ticket?
         [Rewritten query → What was the resolution for Ticket ID TICK-001 regarding the authentication failures after a password reset?]
         Assistant: The resolution for Ticket ID TICK-001 was as follows:

1.